In [1]:
import random
import operator
import pandas as pd
from tabulate import tabulate 

In [2]:
"""
A class that implements arithmetic operations on integers represented
by symbolic sequences
"""

class Path:
    def __init__(self, symbols=None, number=None):
        valid_chars = {'S', 'P'}

        if symbols is not None and number is not None:
            raise ValueError('Uncertainty in the way the path in definition')
        elif symbols is not None:
            if all(char in valid_chars for char in symbols):
                self.symbols = symbols if symbols else []
            else:
                raise ValueError(f'Only {valid_chars} in {symbols} allowed')
        elif number is not None:
            if isinstance(number, int):
                self.symbols = self.from_int(number)
            else:
                raise ValueError(f'Only int number allowed')
        else:
            self.symbols = []
    
    def from_int(self, n):
        """int -> Path convertation"""
        return ['S' if n > 0 else 'P' for _ in range(n)]

    def normalize(self):
        """Removes adjacent SP/PS pairs"""
        stack = []
        for s in self.symbols:
            if stack and ((stack[-1] == 'S' and s == 'P') or (stack[-1] == 'P' and s == 'S')):
                stack.pop()
            else:
                stack.append(s)
        return Path(stack)

    def simplify(self, path):
        """
        Simplifies the path by removing all loops. 
        Returns either an empty path or a path of identical characters.
        """

        result_list = list(path)  # Convert the string to a list for easy removal
        index = 0
        while index < len(result_list) - 1:
            if result_list[index] != result_list[index + 1]:
                result_list.pop(index + 1)  # Remove the second character
                result_list.pop(index)      # Remove the first character
                index = 0  # Go back to the beginning to handle possible new pairs
            else:
                index += 1  # Move to the next character

        return Path("".join(result_list))  # Convert the list back to a string

    def __add__(self, other):
        """Addition: concatenation"""
        return Path(self.symbols + other.symbols)

    def __sub__(self, b):
        """Substraction"""
        neg_b = b.__neg__()
        return Path(self.symbols + neg_b.symbols)

    def __mul__(self, b):
        """Multiplication"""
        return Path(self.mult_paths(self.symbols, b.symbols))
    
    def __neg__(self):
        """Inversion: reverse order and replacement of S ↔ P"""
        return Path("".join([ 'P' if s == 'S' else 'S' for s in reversed(self.symbols)]))

    def inv(self, path):
        """Inverts the Path (replaces 'S' with 'P' and vice versa)"""
        return "".join([('P' if c == 'S' else 'S') for c in path])

    def reverse(self, path):
        """Reverse the Path"""
        return path[::-1]

    def simplify_path(self, path):
        """Simplifies the path by removing adjacent reverse elements"""
        result = []
        for char in path:
            if result and (
                (result[-1] == 'S' and char == 'P') or
                (result[-1] == 'P' and char == 'S')
            ):
                result.pop()
            else:
                result.append(char)
        return "".join(result)

    def path_sum(self, path):
        """Вычисляет сумму пути (a = +1, b = -1)."""
        return sum([1 if c == 'S' else -1 for c in path])
    

    def abs(self, path):
        return "".join(self.reverse(self.inv(path))) if self.path_sum(path) < 0 else "".join(path)
    
    def mult_paths(self, a, b):
        dq = ''
        for _ in range(abs(self.path_sum(a))):
            dq += b

        if self.path_sum(a) < 0:
            dq = self.inv(dq)
        
        return dq  

    def divide(self, divisor):
        """Divides the self.symbols path into the divisor path with the remainder"""
    
        # The case when both numbers have the same sign (both positive or both negative)
        dividend_num = self.path_sum(self.symbols)
        divisor_num = self.path_sum(divisor.symbols)

        # Division by zero
        if divisor_num == 0:
            # Calculating the entropy of the divisor
            divisor_entropy = self.entropy(divisor.symbols)
            
            # We determine the quotient based on entropy
            # The higher the entropy, the more "information" there is in the divider.
            quotient_value = int(divisor_entropy * 10) + 1  # +1, чтобы избежать нулевого значения
            
            # Taking into account the divisible sign
            quotient = 'S' * quotient_value if self.path_sum(self.symbols) >= 0 else 'P' * quotient_value
            
            # Calculating the remainder
            remainder = self.symbols
            delta = self.reverse(self.inv(divisor.symbols))
            for _ in range(quotient_value):
                remainder = remainder + delta
            
            return Path(quotient), Path(remainder)
        
        if (dividend_num >= 0 and divisor_num > 0) or (dividend_num <= 0 and divisor_num < 0):
            abs_dividend = self.abs(self.symbols)
            abs_divisor = self.abs(divisor.symbols)
            
            quotient = ''
            remainder = abs_dividend
            delta = self.reverse(self.inv(abs_divisor))
            
            while self.path_sum(remainder) >= abs(divisor_num):
                remainder = self.simplify_path(remainder + delta)
                quotient += 'S'
            
            # Restoring the signs
            if dividend_num < 0 and divisor_num < 0:
                remainder = self.inv(remainder)
            
            return Path(quotient), Path(remainder)
        
        # The case when the numbers have different signs
        else:
            abs_divisor = self.abs(divisor.symbols)
            
            # We find the largest multiple of the divisor that does not exceed the divisible
            quotient = ''
            
            if dividend_num > 0 and divisor_num < 0:
                # The divisible is positive, the divisor is negative
                while dividend_num - (self.path_sum(quotient) + 1) * divisor_num > abs(self.path_sum(divisor.symbols)):
                    quotient += 'P'
                
                dq = self.mult_paths(quotient, divisor.symbols)
                remainder = self.symbols + self.reverse(self.inv(dq))  
                
            else:  # dividend < 0 and divisor > 0
                # The divisible is negative, the divisor is positive
                while dividend_num - (self.path_sum(quotient) + 1) * divisor_num < -abs(self.path_sum(divisor.symbols)):
                    quotient += 'P'
                
                dq = self.mult_paths(quotient, divisor.symbols)
                remainder = self.symbols + self.reverse(self.inv(dq))  
            
            return Path(quotient), Path(remainder)

    def entropy(self, path):
        """Calculates the entropy of a sequence as a measure of its complexity"""
        if not path:
            return 0
        
        # Counting the number of switches between S and P
        switches = 0
        for i in range(1, len(path)):
            if path[i] != path[i-1]:
                switches += 1
        
        # We calculate entropy as the ratio of the number of switches to the length
        return switches / len(path)

    def __floordiv__(self, divisor):
        """Redefining an operation //"""
        return self.divide(divisor)[0]

    def __mod__(self, divisor):
        """Redefining an operation %"""
        return self.divide(divisor)[1]
    
    def __repr__(self):
        return ''.join(self.symbols)
    
    def __int__(self):
        simple_path = self.simplify(self.symbols)
        n = len(simple_path.symbols)
        
        if n == 0:
            return 0
        elif simple_path.symbols[0] == 'S':
            return n
        else:
            return -n        
        

In [3]:
def is_loop(path):
    """Returns True if the path is a loop"""
    return int(path) == 0

def generate_random_paths(length):
    """Generating random paths of length"""
    generated_string = ''.join(random.choice(['S', 'P']) for _ in range(length))
    return Path(generated_string)

def generate_random_loops(length):
    """Generating random loops of length"""
    if length % 2 != 0:
        generated_string = ''  # String length must be even
    else:
        half_length = length // 2
        character_list = ['S'] * half_length + ['P'] * half_length
        random.shuffle(character_list)
        generated_string = "".join(character_list)
    return Path(generated_string)

In [4]:
def random_numbers_divide(a_rnd_generator, b_rnd_generator, a_path_len, b_path_len, num):
    results = []

    for _ in range(num):
        a_p = a_rnd_generator(a_path_len)
        b_p = b_rnd_generator(b_path_len)
        a_div_b = a_p // b_p
        a_mod_b = a_p % b_p
        ex = {
            'a': int(a_p),
            'a (path)': a_p,
            'b': int(b_p),
            'b (path)': b_p,
            'a // b': int(a_div_b),
            'a mod b': int(a_mod_b),
            'a // b (path)': a_div_b,
            'a mod b (path)': a_mod_b
        }
        results.append(ex)   

    return pd.DataFrame(results) 

In [5]:
def random_numbers_operation(operation, a_rnd_generator, b_rnd_generator, a_path_len, b_path_len, num):
    results = []

    for _ in range(num):
        a_p = a_rnd_generator(a_path_len)
        b_p = b_rnd_generator(b_path_len)
        a_plus_b = operation(a_p, b_p)
        ex = {
            'a': int(a_p),
            'a (path)': a_p,
            'b': int(b_p),
            'b (path)': b_p,
            'a + b': int(a_plus_b),
            'a + b (path)': a_plus_b
        }
        results.append(ex)   

    return pd.DataFrame(results) 

In [6]:
# Adding 10 different random pairs of non-zero numbers
results = random_numbers_operation(operator.add, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)         b  b (path)      a + b  a + b (path)
---  -------------  ---  ----------  -------  ----------------------
 -1  SPSPPPPSSSSPP    1  PPSPPSSSS         0  SPSPPPPSSSSPPPPSPPSSSS
 -3  PPPSPSPSSPPSP    3  PSSSPSSSP         0  PPPSPSPSSPPSPPSSSPSSSP
 -9  PPPPPPPSSPPPP   -3  PPSSPPPPS       -12  PPPPPPPSSPPPPPPSSPPPPS
 -5  PPPPSSPPPSSPP   -5  PPPSSPPPP       -10  PPPPSSPPPSSPPPPPSSPPPP
  3  PSSSSSSPPPSPS    3  SPPPSSSSS         6  PSSSSSSPPPSPSSPPPSSSSS
 -3  PSSPPPSPSPPPS   -1  SSPPPPSPS        -4  PSSPPPSPSPPPSSSPPPPSPS
 -1  SPSPSSSPPPSPP   -1  SPPSSPPPS        -2  SPSPSSSPPPSPPSPPSSPPPS
  1  SPPSSSPSPSSPP    3  PPPSSSSSS         4  SPPSSSPSPSSPPPPPSSSSSS
 -1  PPSPSPPSSSSPP   -1  PPSSPPPSS        -2  PPSPSPPSSSSPPPPSSPPPSS
  3  SSPSSSPSPSSPP    5  SSPSSPSSS         8  SSPSSSPSPSSPPSSPSSPSSS


In [7]:
# Subtracting 10 different random pairs of non-zero numbers
results = random_numbers_operation(operator.sub, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)         b  b (path)      a + b  a + b (path)
---  -------------  ---  ----------  -------  ----------------------
  5  SSSSSPSSPSSPP   -1  PSPPSPPSS         6  SSSSSPSSPSSPPPPSSPSSPS
  3  SPPSSPSPPSSSS   -3  PSSPPPSPP         6  SPPSSPSPPSSSSSSPSSSPPS
 -5  PPSPPSPPPPPSS   -1  PPSPPPSSS        -4  PPSPPSPPPPPSSPPPSSSPSS
  7  PSSPSSSSPSSSS    5  PSSSSPSSS         2  PSSPSSSSPSSSSPPPSPPPPS
 -3  SSPPPPPPSSPPS    1  SPSPSPSPS        -4  SSPPPPPPSSPPSPSPSPSPSP
 -1  SSSPSSPPSPPPP    1  PSSSSPSPP        -2  SSSPSSPPSPPPPSSPSPPPPS
 -1  SPPSPSSSSPPPP   -1  PSSSPSPPP         0  SPPSPSSSSPPPPSSSPSPPPS
 -1  PSPSSPPPSSPPS    1  PSSSSPPSP        -2  PSPSSPPPSSPPSSPSSPPPPS
 -1  SSPPSPPPPPSSS   -1  SPSPPPSPS         0  SSPPSPPPPPSSSPSPSSSPSP
  5  SPSSPSSPPSSSS    7  SSSPSSSSS        -2  SPSSPSSPPSSSSPPPPPSPPP


In [8]:
# Multiplication of 10 different random pairs of non-zero numbers
results = random_numbers_operation(operator.mul, generate_random_paths, generate_random_paths, 13, 9, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)         b  b (path)      a + b  a + b (path)
---  -------------  ---  ----------  -------  ---------------------------------------------------------------
  3  PPSSSPSPSSSPS    5  SSPSSPSSS        15  SSPSSPSSSSSPSSPSSSSSPSSPSSS
 -1  SPSPSSPPPSPSP    3  SSPSPSSPS        -3  PPSPSPPSP
  1  SPPPPSPSSPSSS   -3  SPPPPSPPS        -3  SPPPPSPPS
 -3  PPSPPPSSPPSPS    3  PSSSPPSSS        -9  SPPPSSPPPSPPPSSPPPSPPPSSPPP
 -3  SPSSPPPPPSPPS   -3  PPPSPPSSP         9  SSSPSSPPSSSSPSSPPSSSSPSSPPS
  1  PSSPSSSPPSPPS   -7  PSPPPPPPP        -7  PSPPPPPPP
 -1  PSPSPPSSSPPPS   -3  SPPPPPSPS         3  PSSSSSPSP
 -1  PSPSPSSPPSSPP    1  SPSPSPSSP        -1  PSPSPSPPS
  7  SSSSPSPSSSSPS    3  PSSSSSPSP        21  PSSSSSPSPPSSSSSPSPPSSSSSPSPPSSSSSPSPPSSSSSPSPPSSSSSPSPPSSSSSPSP
  1  PSSPSPSSPPSPS   -5  SPPPSPPPP        -5  SPPPSPPPP


In [9]:
# Dividing 10 different random pairs of non-zero numbers
results = random_numbers_divide(generate_random_paths, generate_random_paths, 11, 15, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)       b  b (path)           a // b    a mod b  a // b (path)    a mod b (path)
---  -----------  ---  ---------------  --------  ---------  ---------------  --------------------------
  3  SSPSSPPPSSS    3  PSPSPPPSSSSPSSS         1          0  S
  3  PPSSPSSSPSS    3  PPPPSPSSSSPSSSS         1          0  S
 -1  PPSPSPSSPPS   -1  PSPPSSPPPSPSPSS         1          0  S
  3  SPSSSSPPPSS    1  SSPSPPSSSSSPPPP         3          0  SSS
 -1  PPSSPPPSSSP    5  SSSPPSSSPPSPSSS        -1          4  P                PPSSPPPSSSPSSSPSPPSSSPPSSS
  1  SSPSSPPPSPS    1  PSSSSPSPPSPPSSP         1          0  S
 -5  PSSPPPSPPPP    5  PPSSSSPSSSSSPSP        -1          0  P                PSSPPPSPPPPPSPSSSSSPSSSSPP
  3  SPSPPSSPSSS   -3  PPPPSPSPPSPPSSS        -1          0  P                SPSPPSSPSSSSSSPPSPPSPSPPPP
  1  SPSSPSPPSPS    7  SPPSSSSSSSSPPSS         0          1                   SPSSPSPPSPS
  3  PPSSSSSPPSS   -3  SPPSPPPPSSPPSPS        -1          0  P                PP

In [10]:
# Dividing 10 different random non-zero numbers by 10 random zeros with different structure
results = random_numbers_divide(generate_random_paths, generate_random_loops, 9, 14, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)      b  b (path)          a // b    a mod b  a // b (path)    a mod b (path)
---  ----------  ---  --------------  --------  ---------  ---------------  -------------------------------------------------------------------------------------------------------------------------
  3  SSPPPSSSS     0  PPSPPSPSPSSPSS         7          3  SSSSSSS          SSPPPSSSSPPSPPSPSPSSPSSPPSPPSPSPSSPSSPPSPPSPSPSSPSSPPSPPSPSPSSPSSPPSPPSPSPSSPSSPPSPPSPSPSSPSSPPSPPSPSPSSPSS
 -1  PPPPSSSSP     0  SSPSPSSPPSPSPP        -7         -1  PPPPPPP          PPPPSSSSPSSPSPSSPPSPSPPSSPSPSSPPSPSPPSSPSPSSPPSPSPPSSPSPSSPPSPSPPSSPSPSSPPSPSPPSSPSPSSPPSPSPPSSPSPSSPPSPSPP
  1  PPSSSPSSP     0  SSPSPSPPSPSPPS         8          1  SSSSSSSS         PPSSSPSSPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPPPSSPSPSSPSPSPP
 -3  PPSSPPPSP     0  PSSPSPPPSSSPSP        -6         -3  PPPPPP           PPSSPPPSPSPSPPPSSSPSPPSSPSPPPSSSPSPPSSPSPPPSSSPSPPSSPSPPPSSSPSPPSSPSP

In [11]:
# Dividing 10 different random pairs of zeros with different structure
results = random_numbers_divide(generate_random_loops, generate_random_loops, 6, 14, 10)
print(tabulate(results, showindex=False, headers=results.columns))

  a  a (path)      b  b (path)          a // b    a mod b  a // b (path)    a mod b (path)
---  ----------  ---  --------------  --------  ---------  ---------------  --------------------------------------------------------------------------------------------------------
  0  PPSPSS        0  PPSSSPPSSPSPSP         6          0  SSSSSS           PPSPSSSPSPSPPSSPPPSSSPSPSPPSSPPPSSSPSPSPPSSPPPSSSPSPSPPSSPPPSSSPSPSPPSSPPPSSSPSPSPPSSPPPSS
  0  SPSSPP        0  PPSSPPSSPSSPPS         6          0  SSSSSS           SPSSPPPSSPPSPPSSPPSSPSSPPSPPSSPPSSPSSPPSPPSSPPSSPSSPPSPPSSPPSSPSSPPSPPSSPPSSPSSPPSPPSSPPSS
  0  SSPPSP        0  SPPPPSSSSSPPSP         4          0  SSSS             SSPPSPSPSSPPPPPSSSSPSPSSPPPPPSSSSPSPSSPPPPPSSSSPSPSSPPPPPSSSSP
  0  SSPSPP        0  PSSSSPPPSSSPPP         3          0  SSS              SSPSPPSSSPPPSSSPPPPSSSSPPPSSSPPPPSSSSPPPSSSPPPPS
  0  PSPPSS        0  SSPPPPPPSSSPSS         3          0  SSS              PSPPSSPPSPPPSSSSSSPPPPSPPPSSSSSSPPPPSPPPSSSSSSPP
  0  